In [ ]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [ ]:
df_HV_HH = pd.read_csv('valid_HV_HH_24h_tol2_drift_pairs_with_errors.csv')

In [ ]:
import pandas as pd
from pathlib import Path

# Ensure buoy times are datetime
df_HV_HH["t0"] = pd.to_datetime(df_HV_HH["t0"])
df_HV_HH["t1"] = pd.to_datetime(df_HV_HH["t1"])

# Function to extract SAR timestamp from TIFF path
def extract_sar_time(path):
    filename = Path(path).stem  # e.g. '20190424T0653'
    return pd.to_datetime(filename, format="%Y%m%dT%H%M")

# Extract SAR timestamps
df_HV_HH["sar_t0"] = df_HV_HH["tiff0_path"].apply(extract_sar_time)
df_HV_HH["sar_t1"] = df_HV_HH["tiff1_path"].apply(extract_sar_time)

# Compute time differences in hours (signed)
df_HV_HH["buoy_SAR_dt0_hours"] = (
    (df_HV_HH["t0"] - df_HV_HH["sar_t0"]).dt.total_seconds() / 3600
)

df_HV_HH["buoy_SAR_dt1_hours"] = (
    (df_HV_HH["t1"] - df_HV_HH["sar_t1"]).dt.total_seconds() / 3600
)

# Absolute differences
df_HV_HH["buoy_SAR_dt0_abs_hours"] = df_HV_HH["buoy_SAR_dt0_hours"].abs()
df_HV_HH["buoy_SAR_dt1_abs_hours"] = df_HV_HH["buoy_SAR_dt1_hours"].abs()


In [ ]:
df_HV_HH = df_HV_HH[(df_HV_HH["buoy_SAR_dt0_abs_hours"] <= 0.5) & (df_HV_HH["buoy_SAR_dt1_abs_hours"] <= 0.5)]


In [ ]:
df_HV_HH.columns

In [ ]:
df_HV_HH = df_HV_HH[df_HV_HH['n_pm']>400]


In [ ]:
df_HV_HH['rel_endpoint_err'].notna().sum()

In [ ]:
df_HV_HH = df_HV_HH[df_HV_HH['rel_endpoint_err'].notna()]

In [ ]:
# df_HV_HH[df_HV_HH['endpoint_err_m']> 5000].iloc[6]

In [ ]:
# df_HV_HH = df_HV_HH[df_HV_HH['rel_endpoint_err'] < 10]

In [ ]:
len(df_HV_HH)

In [ ]:
df_HV_HH['rel_endpoint_err'].describe()

In [ ]:
df_HV_HH['rel_endpoint_err'].hist(bins=100)

In [ ]:
df_HV_HH['angle_error_deg'].describe()

In [ ]:
df_HV_HH['angle_error_deg'].hist(bins=50)

In [ ]:
df_HV_HH['abs_mag_error_m'].describe()

In [ ]:
df_HV_HH['abs_mag_error_m'].hist(bins=100)

In [ ]:
df_HV_HH['endpoint_err_m'].describe()

In [ ]:
df_HV_HH['endpoint_err_m'].hist(bins=100)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

sns.set_style("whitegrid")
sns.set_palette("colorblind")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def scatter_with_1to1(ax, x, y, xlabel, ylabel, title):
    # Correlation
    r, _ = pearsonr(df_HV_HH[x].to_numpy(), df_HV_HH[y].to_numpy())

    sns.scatterplot(data=df_HV_HH, x=x, y=y, ax=ax, s=35, alpha=0.7)

    # 1:1 line + equal limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    lo = min(xlim[0], ylim[0])
    hi = max(xlim[1], ylim[1])
    ax.plot([lo, hi], [lo, hi], "--", color="gray", linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.text(
        0.05, 0.95, f"r = {r:.3f}",
        transform=ax.transAxes, va="top",
        fontsize=11,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

# Top-left: dx
scatter_with_1to1(
    axes[0, 0],
    x="buoy_dx_m", y="sar_dx_m",
    xlabel="Buoy dx (m)", ylabel="SAR dx (m)",
    title="dx: Buoy vs SAR"
)

# Top-right: dy
scatter_with_1to1(
    axes[0, 1],
    x="buoy_dy_m", y="sar_dy_m",
    xlabel="Buoy dy (m)", ylabel="SAR dy (m)",
    title="dy: Buoy vs SAR"
)

# --- Bottom-left: component errors (buoy - SAR) with mean/std annotation ---
err_dx = (df_HV_HH["buoy_dx_m"] - df_HV_HH["sar_dx_m"]).to_numpy()
err_dy = (df_HV_HH["buoy_dy_m"] - df_HV_HH["sar_dy_m"]).to_numpy()

mean_x, mean_y = np.nanmean(err_dx), np.nanmean(err_dy)
std_x,  std_y  = np.nanstd(err_dx),  np.nanstd(err_dy)
r_err, _ = pearsonr(np.nan_to_num(err_dx, nan=0.0), np.nan_to_num(err_dy, nan=0.0))

sns.scatterplot(x=err_dx, y=err_dy, ax=axes[1, 0], s=35, alpha=0.7)

# zero lines + mean lines
axes[1, 0].axhline(0, color="gray", linewidth=1)
axes[1, 0].axvline(0, color="gray", linewidth=1)
axes[1, 0].axhline(mean_y, color="gray", linestyle="--", linewidth=1)
axes[1, 0].axvline(mean_x, color="gray", linestyle="--", linewidth=1)

axes[1, 0].set_xlabel("dx error = buoy - SAR (m)")
axes[1, 0].set_ylabel("dy error = buoy - SAR (m)")
axes[1, 0].set_title("Component errors (buoy - SAR)")

axes[1, 0].text(
    0.05, 0.95,
    f"r = {r_err:.3f}\n"
    f"mean_x = {mean_x:.2f} m, std_x = {std_x:.2f} m\n"
    f"mean_y = {mean_y:.2f} m, std_y = {std_y:.2f} m",
    transform=axes[1, 0].transAxes, va="top",
    fontsize=10,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)


# --- Bottom-right: Relative endpoint error (log distribution) ---
rel_err = (
    df_HV_HH["rel_endpoint_err"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .to_numpy()
)

rel_err = rel_err[rel_err > 0]  # required for log scale

# Linear-space stats
mean_rel = np.mean(rel_err)
std_rel = np.std(rel_err)
n_rel = len(rel_err)

# Log-spaced bins
nbins = 40
bins = np.logspace(np.log10(rel_err.min()),
                   np.log10(rel_err.max()),
                   nbins + 1)

sns.histplot(rel_err, bins=bins, ax=axes[1, 1])

axes[1, 1].set_xscale("log")
axes[1, 1].set_xlabel("Relative endpoint error [log scale]")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Relative endpoint error distribution")

axes[1, 1].text(
    0.05, 0.95,
    f"mean = {mean_rel:.3f}\n"
    f"std = {std_rel:.3f}\n\n"
    f"n = {n_rel}",
    transform=axes[1, 1].transAxes,
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
)




plt.tight_layout()
plt.show()



In [ ]:
df_HV_HH['n_ft'].describe()

In [ ]:
error_cols = [
    "abs_mag_error_m",
    "angle_error_deg",
    "endpoint_err_m",
    "rel_endpoint_err",
]


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, col in zip(axes, error_cols):
    sns.violinplot(
        y=df_HV_HH[col],
        inner=None,
        cut=0,
        ax=ax
    )
    sns.boxplot(
        y=df_HV_HH[col],
        width=0.2,
        boxprops={"facecolor":"white"},
        showfliers=True,
        ax=ax
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()


In [ ]:
cols_corr = ["rel_endpoint_err", "n_ft", "n_pm"]
df_corr = df_HV_HH[cols_corr].dropna()


In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

pairs = [("n_ft", axes[0]), ("n_pm", axes[1])]

for xcol, ax in pairs:
    x = df_corr[xcol]
    y = df_corr["rel_endpoint_err"]

    r = np.corrcoef(x, y)[0, 1]

    ax.scatter(x, y, alpha=0.6)
    ax.set_xlabel(xcol)
    ax.set_title(f"rel_endpoint_err vs {xcol}\nPearson r = {r:.2f}")
    ax.grid(True)

axes[0].set_ylabel("rel_endpoint_err")
plt.tight_layout()
plt.show()


In [ ]:
df_corr.corr()


In [ ]:
sns.heatmap(df_corr.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()
